In [ ]:
# 1. Clona il repository
!git clone https://github.com/MisterCioffi/SteganoGAN.git

# 2. Entra nella cartella del progetto
%cd SteganoGAN

# 3. Rimuovi i limiti superiori stringenti dal file setup.py
with open('setup.py', 'r') as f:
    setup_content = f.read()

# Rimuoviamo i vincoli superiori che rompono Colab
setup_content = setup_content.replace('<2.5.0', '')
setup_content = setup_content.replace('<1.2.0', '')
setup_content = setup_content.replace('<1.16.0', '')
setup_content = setup_content.replace('<8.0.0', '')
setup_content = setup_content.replace('<2.0.0', '')

with open('setup.py', 'w') as f:
    f.write(setup_content)

print("setup.py patchato con successo!")

Cloning into 'SteganoGAN'...
remote: Enumerating objects: 1964, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 1964 (delta 5), reused 15 (delta 3), pack-reused 1943 (from 1)
Receiving objects: 100% (1964/1964), 43.51 MiB | 23.93 MiB/s, done.
Resolving deltas: 100% (1015/1015), done.
/content/SteganoGAN
setup.py patchato con successo!


In [2]:
# Installa le dipendenze e il pacchetto
!pip install -e .

Obtaining file:///content/SteganoGAN
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for reedsolo: filename=reedsolo-0.3-py3-none-any.whl size=5301 sha256=170a658070b4fd0ef68faf83ee2f969691976ff95a433c444c23db0a41984b3a
  Stored in directory: /root/.cache/pip/wheels/5c/14/ad/f69c34ef121e2c2ef5ae1123d4efaec9f7e246ad734186ffa3
Successfully built reedsolo
  Running setup.py develop for steganogan


In [ ]:
import os
import torch
import torch.optim
import torch.optim.adam
import steganogan.models
from steganogan import SteganoGAN

# --- 1. PATCH PER L'OTTIMIZZATORE ADAM (Livello 1 e 2) ---
_original_optimizer_setstate = torch.optim.Optimizer.__setstate__
def _safe_setstate(self, state):
    if isinstance(state, dict) and 'defaults' not in state:
        state['defaults'] = {}
    try:
        _original_optimizer_setstate(self, state)
    except Exception:
        pass
torch.optim.Optimizer.__setstate__ = _safe_setstate

# NOVITÀ: Patch specifica per Adam per l'errore param_groups
_original_adam_setstate = torch.optim.Adam.__setstate__
def _safe_adam_setstate(self, state):
    if not hasattr(self, 'param_groups'):
        self.param_groups = []  # Gli diamo una lista vuota per non farlo arrabbiare
    try:
        _original_adam_setstate(self, state)
    except Exception:
        pass
torch.optim.Adam.__setstate__ = _safe_adam_setstate


# --- 2. PATCH PER LA SICUREZZA DI PYTORCH 2.6+ ---
@classmethod
def custom_load(cls, architecture=None, path=None, cuda=True, verbose=False):
    if architecture and not path:
        model_name = '{}.steg'.format(architecture)
        pretrained_path = os.path.join(os.path.dirname(steganogan.models.__file__), 'pretrained')
        path = os.path.join(pretrained_path, model_name)
    elif (architecture is None and path is None) or (architecture and path):
        raise ValueError('Please provide either an architecture or a path to pretrained model.')

    # weights_only=False per bypassare il blocco di sicurezza
    steganogan_model = torch.load(path, map_location='cpu', weights_only=False)
    steganogan_model.verbose = verbose

    steganogan_model.encoder.upgrade_legacy()
    steganogan_model.decoder.upgrade_legacy()
    steganogan_model.critic.upgrade_legacy()

    steganogan_model.set_device(cuda)
    return steganogan_model

SteganoGAN.load = custom_load


# --- 3. IL TEST FINALE ---
print("Caricamento del modello (con Super Patch 2.0 attiva)...")
steganogan_model = SteganoGAN.load(architecture='dense')

# Ora proviamo finalmente il Logo!
cover_image_path = 'research/input.png'
stego_image_path = 'output.png'
messaggio_segreto = "https://it.wikipedia.org/wiki/Pongo_pygmaeus"

print("\nNascondo il messaggio nel Logo...")
steganogan_model.encode(cover_image_path, stego_image_path, messaggio_segreto)
print(f"Immagine salvata in: {stego_image_path}")

print("\nEstraggo il messaggio dal Logo modificato...")
messaggio_decodificato = steganogan_model.decode(stego_image_path)

print("\n--- RISULTATO ---")
print(f"Messaggio estratto: {messaggio_decodificato}")

Caricamento del modello (con Super Patch 2.0 attiva)...

Nascondo il messaggio nel Logo...
Immagine salvata in: output_orango.png

Estraggo il messaggio dal Logo modificato...

--- RISULTATO ---
Messaggio estratto: https://it.wikipedia.org/wiki/Pongo_pygmaeus
